# 下载 Llava-1.5-7b 模型并进行推理

按照 [Hugging Face transformers 库文档](https://huggingface.co/docs/transformers/main/en/model_doc/llava)，演示如何通过 transformers 库使用 Llava-1.5-7b 进行推理。

## 环境配置

本 notebook 在 AutoDL 上使用 PyTorch / 2.3.0 / 3.12(ubuntu22.04) / 12.1 镜像环境跑通，注意需要使用 4.57.6 版本的 transformers，高版本有概率不适配 pytorch-2.3.0。

In [1]:
# pip install transformers==4.57.6 huggingface_hub modelscope

## 加载依赖

Hugging Face 的 transformers 库已经对 Llava 模型做了适配。我们直接使用实现好的模块。

In [2]:
import requests
from PIL import Image

import torch
from transformers import AutoProcessor, LlavaForConditionalGeneration, LlavaProcessor

## 下载模型

在 python 环境中使用命令行运行下面的代码：安装库（已经安装就不用再装一遍）、切换下载源为镜像源、下载模型到本地指定路径。

```shell
pip install -U huggingface_hub
export HF_ENDPOINT=https://hf-mirror.com
huggingface-cli download --resume-download llava-hf/llava-1.5-7b-hf --local-dir llava-1.5-7b-hf --local-dir-use-symlinks False
```

也可以在 notebook 中完成下载。本 notebook 默认使用魔塔社区在 notebook 中完成下载，具体可以直接往下看本节的最后一部分。

### 命令逐段解析

```bash
huggingface-cli download --resume-download llava-hf/llava-1.5-7b-hf --local-dir llava-1.5-7b-hf --local-dir-use-symlinks False
```

*   **`huggingface-cli`**：这是 Hugging Face 官方的命令行工具。
*   **`download`**：下载指令。
*   **`--resume-download`**：**断点续传**。如果下载中途中断，再次运行命令时会从断开的地方继续，不用从头开始。
*   **`llava-hf/llava-1.5-7b-hf`**：这是模型在 Hugging Face Hub 上的**仓库 ID**（`用户名/模型名`），指定了要下载哪个模型。
*   **`--local-dir llava-1.5-7b-hf`**：**指定本地保存目录**。它告诉工具，不要用默认的隐藏缓存路径，而是将模型文件直接放进 `llava-1.5-7b-hf` 这个文件夹里。
*   **`--local-dir-use-symlinks False`**：**禁用符号链接**。这个参数设为 `False` 时，工具会**复制真实的模型文件**到目标目录，而不是创建指向默认缓存（`~/.cache/huggingface/hub`）的快捷方式（符号链接）。这样做的好处是得到的 `llava-1.5-7b-hf` 文件夹是**完全独立**的，可以随意移动、复制或打包，不用担心破坏了与隐藏缓存的链接关系。

### 模型会下载到哪个路径？

模型会下载到**当前终端所在的工作目录**下的 `llava-1.5-7b-hf` 文件夹中。

因为提供的 `--local-dir` 参数值 `llava-1.5-7b-hf` 是一个**相对路径**。例如，如果你在 `/home/user/projects` 目录下运行此命令，模型最终的存放路径就是：
`/home/user/projects/llava-1.5-7b-hf/`

如果你希望保存到其他固定位置，可以将 `--local-dir` 的值改为一个**绝对路径**，例如：`--local-dir /data/models/llava-1.5-7b-hf`。

### 常见问题

如果命令行提示找不到 huggingface-cli 命令，是环境变量没设置好导致的。自行找相关的教程，把对应 python 环境下的 Scripts 文件夹添加进系统环境变量。完成后开一个新的 shell 窗口即可。

如果还不行，24 年（0.22.0版本）后的 huggingface-hub 不再推荐使用旧版命令，更换为使用下面的命令下载。

```shell
hf download llava-hf/llava-1.5-7b-hf --local-dir llava-1.5-7b-hf
```

在 AutoDL 上运行时，这样设置环境变量，将缓存和下载路径切换到数据盘。

```shell
export HF_ENDPOINT=https://hf-mirror.com
export HF_HOME=/root/autodl-tmp/cache/
export HF_HUB_DISABLE_XET=1

hf download llava-hf/llava-1.5-7b-hf --local-dir /root/autodl-tmp/llava-1.5-7b-hf
```

### 下载太慢

如果是国内网络环境，建议使用魔塔社区进行下载。本笔记本默认使用魔塔社区进行下载，参考下方的代码块。

如果镜像站下载很慢，推荐使用 AutoDL 学术加速从 HuggingFace 主站下载。

```shell
unset HF_ENDPOINT
source /etc/network_turbo
hf download llava-hf/llava-1.5-7b-hf --local-dir /root/autodl-tmp/llava-1.5-7b-hf
```

In [3]:
use_modelscope = True

In [4]:
if use_modelscope:
    # 安装或更新 modelscope 库（如果环境中没有的话）
    # pip install -U modelscope
    
    # 从魔搭下载模型
    from modelscope import snapshot_download
    
    model_dir = snapshot_download(
        'llava-hf/llava-1.5-7b-hf',       # 魔搭上的模型 ID，与你给的链接一致
        cache_dir='/root/autodl-tmp',      # 强烈建议指向数据盘，避免系统盘被占满
        revision='master'                  # 主分支，通常就是最新的稳定版
    )
    
    print(f"模型已下载到: {model_dir}")

2026-10-18 15:47:45,549 | INFO    | modelscope_hub.download | Downloading 18 files from llava-hf/llava-1.5-7b-hf@master


Downloading:   0%|          | 0/18 [00:00<?, ?file/s]

模型已下载到: /root/autodl-tmp/models/llava-hf--llava-1.5-7b-hf/snapshots/master


## 加载模型

把模型加载到内存，然后加载到显卡。
以 `float16` 精度加载 7B 模型，仅模型权重本身大约需要 14.1 GB 显存。但在实际部署中，算上视觉编码器、运行时激活和KV缓存，单张图片、batch size=1 的推理通常建议准备 15 GB 到 24 GB 显存。

In [ ]:
model_name_or_path = "/root/autodl-tmp/models/llava-hf--llava-1.5-7b-hf/snapshots/master"  # 指向刚才下载好的文件夹
device = "cuda:0"   # 不做检查，默认有 NVIDIA 显卡

model = LlavaForConditionalGeneration.from_pretrained(
    model_name_or_path,
    dtype=torch.float16,
    low_cpu_mem_usage=True,
).to(device)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

## 加载并测试输入处理模块

对于 LLaVA 模型，在训练时给定了严格的格式，所以推理时我们也需要把自己的输入规范到这个格式，便于模型识别。在我们下载的文件中包含了相关的处理模块，只需要加载即可。

In [6]:
processor = LlavaProcessor.from_pretrained(model_name_or_path)
# 等价于：AutoProcessor

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


### 演示输入文本处理

假设我们需要问 LLaVA 图片中都有些什么，即用户输入为图片（另外上传，后面会演示）和一句话。按照 [Hugging Face transformers 库文档](https://huggingface.co/docs/transformers/main/en/model_doc/llava) 我们需要这样组织字段，然后传给 processor 的 apply_chat_template 进行处理。

这里需要注意的是，在 LLaVA 的原始代码库（`haotian-liu/LLaVA`）中，系统提示被定义在对话模板（`conv_templates`）里，是 `Conversation` 类的一个核心字段。

以 LLaVA-1.5 主要使用的 `vicuna_v1` 模板为例，其 `system` 字段被硬编码为：
> **"A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the human's questions."**
> 

这个系统提示会在生成最终 Prompt 时，被 `get_prompt()` 方法自动添加到所有用户消息的最前面。

这里不指定 system 字段的内容，是因为需要模仿 transformers 库文档给出的示例。HF 的 `transformers` 库在实现 LLaVA 的 `chat_template` 时，为了**让演示代码更简洁**，选择性地移除了这个系统提示。

HF 的维护者在其代码仓库的 Pull Request 中明确解释道：
> “从原始 Llava 仓库的推理代码来看，正确的模板是……**技术上我们应该在这里添加系统提示，但我认为为了保持演示的简洁性，可以将其排除。**”

因此，HF 的 `apply_chat_template` 默认行为（直接从 `USER:` 开始）是一个**功能性简化**，并非对原始模型行为的严格复现。但实现中这不会导致明显的问题，体现了 LLM 对输入的鲁棒性。

In [7]:
prompt_text = "What are these?"
messages = [
    {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": prompt_text},
            ]},
]
# 不做 tokenize，便于打印展示
prompt_format = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

print(prompt_format)

USER: <image>
What are these? ASSISTANT:


另外，如果我们需要**使用 HuggingFace 的 LLaVA Processor 的同时，添加一定的 system 字段**，我们需要严格按照下面的格式修改 messages（system 字段也需要严格的带类型的内容部分的格式），否则无法被 processor 识别。

In [8]:
messages_system = [
    {
        "role": "system",
        "content": [{"type": "text", "text": "You are a helpful assistant."}] # 结构化格式
    },
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "What is in this image?"}
        ]
    }
]

# 不做 tokenize，便于打印展示
prompt_format_system = processor.apply_chat_template(
    messages_system, tokenize=False, add_generation_prompt=True
)

print(prompt_format_system)

You are a helpful assistant. USER: <image>
What is in this image? ASSISTANT:


### 输入图片处理

我们从本地将 `000000039769.jpg` 给模型，作为图片输入。

`apply_chat_template` 只负责生成带有 `<image>` 占位符的文本提示。图像像素必须通过 `processor(images=...)` 或 `apply_chat_template(..., images=...)` 显式传入，否则模型将看不到图片。

#### Processor 做了什么？

`LlavaProcessor` 将文本处理和图片处理整合在一个接口中。

在本示例里：

- `text=prompt`：提供已经构造好的提示词。
- `images=raw_image`：提供真实的图片数据。
- `return_tensors="pt"`：要求返回 PyTorch 张量。

处理完成后，`inputs` 通常包含 `input_ids`、`attention_mask` 和 `pixel_values` 等字段。其中，`pixel_values` 保存经过预处理的图片张量，`input_ids` 则表示经过分词的文本输入。

因此，最终得到的 `inputs` 是同时承载文本和视觉输入的模型输入字典。

In [9]:
image_file = "000000039769.jpg"
raw_image = Image.open(image_file)

# 这里的 processor.apply_chat_template 只把 messages 翻译成字符串，即使默认 tokenize=True，
# 真正的 tokenize 过程发生在 processor 的 forward() 中
# 只有在给 apply_chat_template 传入 image 时，这个方法的 tokenize 才会生效
prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(images=raw_image, text=prompt, return_tensors="pt").to(device, torch.float16)

也可以直接传图片得到模型的 inputs。下面的写法更加简洁，但由于兼容性，更推荐上面的写法，让 `processor.apply_chat_template` 只做文本的格式化。

In [ ]:
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    images=raw_image,   # 直接传入 PIL Image
).to(device, dtype=torch.float16)

正如代码中的注释所说，processor.apply_chat_template 是一个多态的方法，在是否已经传入 image 信息时有不同的表现，可以通过函数的文档直接看到这一点：

```python
(method) def apply_chat_template(
    self,
    conversation: Union[list[dict[str, str]], list[list[dict[str, str]]]],
    chat_template: Optional[str] = None,
    **kwargs: Unpack[AllKwargsForChatTemplate],
) -> str:
"""
Similar to the `apply_chat_template` method on tokenizers, this method applies a Jinja template to input conversations to turn them into a single tokenizable string.

The input is expected to be in the following format, where each message content is a list consisting of text and optionally image or video inputs. One can also provide an image, video, URL or local path which will be used to form `pixel_values` when `return_dict=True`. If not provided, one will get only the formatted text, optionally tokenized text.

conversation = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://www.ilankelman.org/stopsigns/australia.jpg"},
            {"type": "text", "text": "Please describe this image in detail."},
        ],
    },
]

Args:
    conversation (`Union[list[Dict, [str, str]], list[list[dict[str, str]]]]`):
        The conversation to format.
    chat_template (`Optional[str]`, *optional*):
        The Jinja template to use for formatting the conversation. If not provided, the tokenizer's chat template is used.
"""
```

processor 中间的处理过程我们获取不到，想观察 tokenizer 的调用结果，可以通过下方的代码手动调用 tokenizer，观察 tokenize 后的 prompt，以及 processor 前向推理对 prompt 和 image 做了怎样的处理，返回的数据结构是什么样的。

**这里的输出仅用于观察文本模板，不包含图像 patch 扩展。真正的多模态输入由 processor.apply_chat_template 或 processor() 的生成**。

In [10]:
# processor.tokenizer.apply_chat_template 这个是纯文本的方法，不需要像上面的依赖于 image
prompt_tokenized = processor.tokenizer.apply_chat_template(messages, add_generation_prompt=True)

print(prompt_tokenized)
print(len(prompt_tokenized))
print(inputs.keys())

[3148, 1001, 29901, 29871, 32000, 29871, 13, 5618, 526, 1438, 29973, 319, 1799, 9047, 13566, 29901]
16
KeysView({'input_ids': tensor([[    1,  3148,  1001, 29901, 29871, 32000, 32000, 32000, 32000, 32000,
         32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000,
         32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000,
         32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000,
         32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000,
         32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000,
         32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000,
         32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000,
         32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000,
         32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000,
         32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000,
     

在较新版本（现在是4.57.6）的 transformers 中，hugging face 实现将复制 num_patches 个 `<image>` 的逻辑（用于创建空位供 image tokens 替换）写到了 `processor` 中（较早版本是在 model 的 forward 中），所以 `inputs_id` 里的文本信息的序列长度会比较上面只用 `apply_chat_template` 中得到的 `prompt_tokenized` 长。在本例中，长度多 $ 576 - 1 = 575 $，但给 input 的处理路径中又添加了 `<bos>`（在 Vocab 中索引为 1，被添加在 input_ids 最前方 ），`inputs_id` 比 `prompt_tokenized` 的长度多 576。

In [11]:
print(inputs["input_ids"].shape)
print(inputs["pixel_values"].shape)

torch.Size([1, 592])
torch.Size([1, 3, 336, 336])


## 调用模型进行推理

generate 代码将自动按照 LLaVA 数据流使用模型进行前向推理。其中包括了使用 ViT 获取 image tokens 并替换掉之前 processor 中填充的 num_patches 个 `<image>` 特殊符号。

`max_new_tokens` 建议设置在 128 以上，过短会导致答案被截断。对于客观视觉问答，推荐 `temperature=0.1~0.2`、`top_p=0.9`；若希望描述更具多样性，可适当提高 `temperature`。

`batch_decode` 返回的序列包含了输入提示词。如需仅查看模型生成的回答，可执行 `generate_ids[:, inputs.input_ids.shape[1]:]` 后再解码。

In [12]:
generate_ids = model.generate(**inputs, max_new_tokens=128)
# generate_ids[:, inputs.input_ids.shape[1]:]
processor.batch_decode(generate_ids, skip_special_tokens=True)

['USER:  \nWhat are these? ASSISTANT: These are two cats lying on a pink couch.']

## 对比使用 system 字段

这里我们使用带 system 的 messages 进行推理，用作对比。

In [ ]:
prompt = processor.apply_chat_template(messages_system, add_generation_prompt=True)
inputs = processor(images=raw_image, text=prompt, return_tensors="pt").to(device, torch.float16)

In [15]:
prompt_tokenized = processor.tokenizer.apply_chat_template(messages_system, add_generation_prompt=True)

print(prompt_tokenized)
print(len(prompt_tokenized))
print(inputs.keys())

[887, 526, 263, 8444, 20255, 29889, 3148, 1001, 29901, 29871, 32000, 29871, 13, 5618, 338, 297, 445, 1967, 29973, 319, 1799, 9047, 13566, 29901]
24
KeysView({'input_ids': tensor([[    1,   887,   526,   263,  8444, 20255, 29889,  3148,  1001, 29901,
         29871, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000,
         32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000,
         32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000,
         32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000,
         32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000,
         32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000,
         32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000,
         32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000,
         32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000,
         32000, 32000, 32000, 32000, 32

In [16]:
generate_ids = model.generate(**inputs, max_new_tokens=128)
# generate_ids[:, inputs.input_ids.shape[1]:]
processor.batch_decode(generate_ids, skip_special_tokens=True)

['You are a helpful assistant. USER:  \nWhat is in this image? ASSISTANT: The image features two cats lying on a pink couch, both of them sleeping.']